# El coste del error: dónde poner el umbral

**Facsímil 7 · Evaluar, calibrar e interpretar** — capítulo 2 (métricas clásicas: matriz de confusión y
coste del error).

La métrica que de verdad importa casi nunca es «el acierto», sino «cuánto me cuesta equivocarme». Y como
los dos tipos de error rara vez cuestan lo mismo, el **umbral** de un clasificador es una **decisión de
negocio**, no un valor sagrado. En este cuaderno dejas de elegir el umbral «por defecto» (0,5) y lo
eliges **por dinero**: en la detección de fraude, dejar pasar un fraude cuesta mucho más que revisar una
transacción legítima de más. Pones número a cada error, dibujas la curva del coste y encuentras el umbral
que la minimiza. Sorpresa: casi nunca es 0,5. Después comprobarás que ese umbral se mueve con los costes,
que cambia de un problema a otro (fraude, spam, cribado médico) y que existe una fórmula que lo predice
cuando las probabilidades son honestas.

### Qué vas a aprender
- Por qué el **acierto** es mala guía cuando una clase es rara y un error cuesta más que otro.
- A leer la **matriz de confusión** y a dibujarla, en términos de coste (falsos negativos frente a falsos
  positivos).
- A **barrer umbrales** y dibujar la curva del coste para encontrar el óptimo.
- Que ese umbral óptimo **depende de tus costes** y casi nunca vale 0,5, y a calcularlo con una fórmula.
- A llevar el mismo razonamiento a **tres problemas cotidianos** distintos y a ver el coste por operación.
- Por qué fiarse de **probabilidades mal calibradas** te da un umbral equivocado.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico. La
gracia no es que «salga», sino entender *por qué* sale.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
np.random.seed(0)

# Fraude: clase rara (8%). Lo normal en estos problemas.
X, y = make_classification(n_samples=6000, weights=[0.92, 0.08], n_informative=6, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
modelo = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
p = modelo.predict_proba(X_te)[:, 1]
print(f"{y_te.sum()} fraudes reales de {len(y_te)} transacciones de prueba ({100*y_te.mean():.1f}%).")
print("El modelo no da un 'si/no', da una PROBABILIDAD de fraude para cada transaccion.")
print("Decidir = elegir a partir de que probabilidad actuamos. Ese corte es el UMBRAL.")


## 1. El acierto engaña con clases raras

Con un 8% de fraude, un modelo tonto que diga «nunca hay fraude» acierta el **92%**... y es completamente
inútil (no caza ni un fraude). El acierto, en problemas desbalanceados, premia ignorar la clase rara. Es
como un detector de incendios que nunca suena: acierta casi siempre, porque casi nunca hay incendio, y
justo falla el día que importa. Lo comprobamos antes de seguir.


In [ ]:
acc_tonto = 1 - y_te.mean()    # "siempre legitima"
acc_modelo = (modelo.predict(X_te) == y_te).mean()
print(f"Modelo tonto ('nunca hay fraude'): acierto {acc_tonto:.3f}  <- altisimo e INUTIL (0 fraudes cazados)")
print(f"Nuestro modelo (umbral 0.5):       acierto {acc_modelo:.3f}")
print(f"Diferencia de acierto: solo {abs(acc_modelo-acc_tonto):.3f}. El acierto apenas distingue al util del inutil.")
print("Con clases raras, el acierto es una guia tramposa. Hay que mirar el COSTE de cada error.")


## 2. La matriz de confusión: los cuatro desenlaces

Antes de hablar de dinero, hay que separar los errores. Toda decisión binaria tiene cuatro casillas: acertar
en cada clase (los dos aciertos) y los dos tipos de error, que tienen nombre propio:

- **Falso negativo (FN):** había fraude y lo dejamos pasar. La transacción se cuela.
- **Falso positivo (FP):** era legítima y la bloqueamos. Molestamos a un cliente honrado.

La **matriz de confusión** cruza lo real con lo predicho y los pone los cuatro a la vista. Con el umbral por
defecto, así queda.


In [ ]:
pred05 = (p >= 0.5).astype(int)
cm = confusion_matrix(y_te, pred05, labels=[1, 0])
vp, fn = cm[0, 0], cm[0, 1]
fp, vn = cm[1, 0], cm[1, 1]
print("                    pred: FRAUDE   pred: LEGITIMA")
print(f"  real FRAUDE          {vp:>4}          {fn:>4}   <- {fn} fraudes que se cuelan (falsos negativos)")
print(f"  real LEGITIMA        {fp:>4}          {vn:>4}   <- {fp} clientes molestados (falsos positivos)")
print(f"\nDe {y_te.sum()} fraudes reales, a 0.5 cazamos {vp} y se nos escapan {fn}.")
print("El acierto mete los dos errores en el mismo saco. La matriz los separa: es el primer paso.")


## 3. La matriz de confusión, dibujada

Un mapa de calor la hace más legible de un vistazo: la diagonal son los aciertos; fuera de ella, los dos
tipos de error. Cuanto más oscura una casilla, más casos caen ahí.


In [ ]:
fig, ax = plt.subplots(figsize=(3.8, 3.4))
ax.imshow(cm, cmap="Greys")
ax.set_xticks([0, 1]); ax.set_xticklabels(["fraude", "legitima"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["fraude", "legitima"])
ax.set_xlabel("predicho"); ax.set_ylabel("real")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
ax.set_title("Matriz de confusion (umbral 0,5)"); plt.tight_layout(); plt.show()


## 4. Ponerle precio a cada error

Aquí está la idea central del cuaderno: los dos errores no cuestan lo mismo.

- **Fraude no detectado** (falso negativo): se pierde el importe. Caro. Digamos **100 €**.
- **Legítima bloqueada** (falso positivo): un cliente molesto, una revisión manual. Barato: **5 €**.

Con esos precios, el coste total de una decisión es `nº de FN × 100 + nº de FP × 5`. Definimos una función
que lo calcula para cualquier umbral: será nuestra brújula el resto del cuaderno.


In [ ]:
COSTE_FN, COSTE_FP = 100, 5

def coste_total(prob, real, u, c_fn=COSTE_FN, c_fp=COSTE_FP):
    pred = (prob >= u).astype(int)
    fn = ((pred == 0) & (real == 1)).sum()
    fp = ((pred == 1) & (real == 0)).sum()
    return fn*c_fn + fp*c_fp

print(f"Coste con el umbral por defecto (0,50): {coste_total(p, y_te, 0.5):>6} EUR")
print("Pregunta del cuaderno: existe un umbral mas barato? Vamos a buscarlo probandolos TODOS.")


## 5. Barrer todos los umbrales

No hay que adivinar: recorremos todos los umbrales posibles de 0,01 a 0,99 y calculamos el coste de cada
uno. El más barato gana. Comparamos el óptimo con el umbral por defecto y vemos cuánto dinero deja sobre la
mesa quedarse con el 0,5 de fábrica.


In [ ]:
umbrales = np.linspace(0.01, 0.99, 99)
costes = np.array([coste_total(p, y_te, u) for u in umbrales])
u_opt = umbrales[int(np.argmin(costes))]
coste_05 = coste_total(p, y_te, 0.5)
print(f"Umbral por defecto (0.50): coste {coste_05:>6} EUR")
print(f"Umbral optimo ({u_opt:.2f}):       coste {int(costes.min()):>6} EUR")
print(f"Ahorro por mover el umbral: {coste_05 - int(costes.min())} EUR ({100*(coste_05-costes.min())/coste_05:.0f}% menos)")


## 6. La curva del coste

Cada punto es un umbral; el eje vertical, lo que te cuesta. El mínimo de esta curva es tu decisión óptima.
Fíjate en que está **lejos de 0,5**: como dejar pasar un fraude es 20 veces más caro que una falsa alarma,
conviene ser desconfiado y bajar el listón para no perder fraudes. A la izquierda del mínimo bloqueas
demasiado (te comes las falsas alarmas); a la derecha dejas pasar fraudes (te comes los importes perdidos).


In [ ]:
plt.figure(figsize=(7, 3.6))
plt.plot(umbrales, costes, color="black")
plt.axvline(u_opt, ls="--", color="black", lw=1, label=f"optimo = {u_opt:.2f}")
plt.axvline(0.5, ls=":", color="#9c9c9c", lw=1, label="por defecto = 0.50")
plt.xlabel("umbral de decision"); plt.ylabel("coste total (EUR)")
plt.title("El umbral mas barato no es 0,5"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 7. Qué cambia entre 0,5 y el óptimo

Mover el umbral no es magia: cambia el reparto de la matriz de confusión. Al bajar el listón al óptimo,
cazamos muchos más fraudes (bajan los falsos negativos) a cambio de molestar a algún cliente legítimo de
más (suben un poco los falsos positivos). Como el fraude es 20 veces más caro, el cambio compensa. Lo
vemos lado a lado.


In [ ]:
for u, etiqueta in [(0.5, "umbral 0,50 (por defecto)"), (u_opt, f"umbral {u_opt:.2f} (optimo)")]:
    pr = (p >= u).astype(int)
    fn_u = ((pr == 0) & (y_te == 1)).sum()
    fp_u = ((pr == 1) & (y_te == 0)).sum()
    vp_u = ((pr == 1) & (y_te == 1)).sum()
    print(f"{etiqueta}")
    print(f"   fraudes cazados: {vp_u:>3} | fraudes colados: {fn_u:>3} | falsas alarmas: {fp_u:>4} "
          f"| coste: {coste_total(p, y_te, u):>6} EUR")
print("\nEl optimo caza muchos mas fraudes; las falsas alarmas suben, pero salen baratas.")


## 8. La fórmula del umbral óptimo

No hace falta barrer a ciegas: si las probabilidades del modelo son **honestas** (calibradas, lo veremos en
el cuaderno de calibración), el umbral que minimiza el coste tiene fórmula. Conviene bloquear cuando el
coste esperado de dejar pasar supera el de bloquear, y eso ocurre justo en:

$$u^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$$

Es decir: cuanto más caro el falso negativo (el fraude), más pequeño el umbral. Comparamos la fórmula con
el óptimo que encontramos barriendo. Si no coinciden del todo, es una pista de que las probabilidades no
están perfectamente calibradas.


In [ ]:
u_teorico = COSTE_FP / (COSTE_FP + COSTE_FN)
print(f"Umbral teorico  (formula):       {u_teorico:.3f}")
print(f"Umbral empirico (barriendo):     {u_opt:.3f}")
print(f"Diferencia: {abs(u_teorico-u_opt):.3f}")
print("\nCaen cerca: la formula da la intuicion y el barrido afina con los datos reales.")
print("Si se separasen mucho, sospecha de las probabilidades (calibracion): ese es el siguiente cuaderno.")


## 9. El umbral óptimo se mueve con los costes

El umbral óptimo no es una propiedad del modelo: depende de **cuánto cuesta cada error**, que es una
decisión de negocio. Si el fraude se vuelve más caro, conviene ser aún más desconfiado (umbral más bajo) a
costa de más falsas alarmas. Lo medimos variando el coste del fraude no detectado y mirando, en cada caso,
el umbral óptimo y cuántos clientes legítimos acabamos molestando.


In [ ]:
print("coste de un fraude | umbral optimo | fraudes que se cuelan | falsas alarmas")
for c_fn in [20, 100, 500, 2000]:
    cs = np.array([coste_total(p, y_te, u, c_fn=c_fn) for u in umbrales])
    u_o = umbrales[int(np.argmin(cs))]
    pred_o = (p >= u_o).astype(int)
    fn_o = ((pred_o == 0) & (y_te == 1)).sum()
    fp_o = ((pred_o == 1) & (y_te == 0)).sum()
    print(f"      {c_fn:>4} EUR      |     {u_o:.2f}     |         {fn_o:>3}          |      {fp_o:>3}")
print("\nCuanto mas caro el fraude, mas baja el umbral: caza mas fraudes a cambio de mas falsas alarmas.")


## 10. El mismo razonamiento, tres problemas distintos

Lo que cambia de un problema a otro es **qué error es el caro**, y eso da la vuelta al umbral. Tres casos
cotidianos, con la misma máquina de antes:

- **Fraude bancario:** dejar pasar fraude es caro (100 €), bloquear una legítima es barato (5 €). Conviene
  ser desconfiado → umbral bajo.
- **Filtro de spam:** que entre un spam es molesto pero barato (1 €); mandar a la papelera un correo
  importante es caro (50 €). Conviene ser prudente al marcar → umbral alto.
- **Cribado médico:** no detectar a un enfermo es gravísimo (200 €); una prueba extra a alguien sano es
  molesta pero asumible (10 €). Conviene no perder a nadie → umbral muy bajo.

El modelo es el mismo; el umbral correcto, no.


In [ ]:
escenarios = {
    "Fraude bancario":  (100, 5),    # (coste FN, coste FP)
    "Filtro de spam":   (1, 50),
    "Cribado medico":   (200, 10),
}
print(f"{'problema':<18}{'FN caro?':<10}{'umbral optimo':<15}{'que prioriza'}")
for nombre, (c_fn, c_fp) in escenarios.items():
    cs = np.array([coste_total(p, y_te, u, c_fn=c_fn, c_fp=c_fp) for u in umbrales])
    u_o = umbrales[int(np.argmin(cs))]
    fn_caro = "si" if c_fn > c_fp else "no"
    prioriza = "no perder positivos" if c_fn > c_fp else "no dar falsas alarmas"
    print(f"{nombre:<18}{fn_caro:<10}{u_o:<15.2f}{prioriza}")
print("\nMismo modelo, mismas probabilidades: el umbral correcto cambia con lo que cuesta cada error.")


## 11. ¿Cuánto cuesta cada transacción de media?

El coste total depende de cuántas transacciones haya. Para comparar entre escenarios o entre meses, es más
honesto el **coste medio por transacción**: el total dividido entre el número de operaciones. Lo calculamos
para el umbral por defecto y para el óptimo del caso de fraude.


In [ ]:
n = len(y_te)
coste_medio_05 = coste_total(p, y_te, 0.5) / n
coste_medio_opt = costes.min() / n
print(f"Coste medio por transaccion, umbral 0,50: {coste_medio_05:.3f} EUR")
print(f"Coste medio por transaccion, umbral optimo: {coste_medio_opt:.3f} EUR")
print(f"Por cada 10.000 transacciones, el umbral optimo ahorra unos {(coste_medio_05-coste_medio_opt)*10000:.0f} EUR.")


## 12. Cuidado: probabilidades mal calibradas estropean el umbral

Todo este cálculo se fía de las probabilidades del modelo. Si están **infladas** (el modelo dice 0,8 cuando
la realidad es 0,5), el umbral que elijas estará sesgado. Lo simulamos a lo bruto: deformamos las
probabilidades como haría un modelo sobreconfiado y recalculamos el umbral óptimo. Verás que se desplaza,
aunque los datos y los costes son los mismos.


In [ ]:
# Modelo "sobreconfiado": empuja las probabilidades hacia los extremos (0 y 1).
p_inflada = p**0.5 / (p**0.5 + (1 - p)**0.5)   # deforma la confianza sin cambiar el orden
costes_infl = np.array([coste_total(p_inflada, y_te, u) for u in umbrales])
u_opt_infl = umbrales[int(np.argmin(costes_infl))]
print(f"Umbral optimo con probabilidades honestas:      {u_opt:.2f}")
print(f"Umbral optimo con probabilidades 'infladas':    {u_opt_infl:.2f}")
print("El orden de las transacciones no ha cambiado, pero el umbral 'correcto' se mueve.")
print("Moraleja: elegir el umbral por coste EXIGE probabilidades calibradas. Ese es el otro cuaderno.")


## 13. Pruébalo tú

1. **Iguala los costes** (`COSTE_FN = COSTE_FP`): vuelve a barrer y comprueba si el óptimo se acerca a 0,5.
   Cuando los errores cuestan lo mismo, el umbral neutro recupera su sentido. Compáralo con la fórmula del
   apartado 8.
2. **Cuenta las falsas alarmas** en el óptimo del fraude: ¿cuántos clientes legítimos molestas para ahorrar
   esos euros? Ese es el otro lado de la balanza que un número no decide por ti.
3. **Inventa tu escenario:** añade a `escenarios` un caso tuyo (moderación de contenidos, aprobar un
   crédito...) con sus dos costes y mira hacia dónde cae el umbral.
4. **Combina con la curva precision-recall:** dibújala (`from sklearn.metrics import precision_recall_curve`)
   y localiza dónde cae tu umbral óptimo. Es otra forma de ver el mismo compromiso.
5. **Deforma más o menos** las probabilidades del apartado 12 (cambia el exponente 0.5 por 0.3 o por 2) y
   observa cómo el umbral óptimo se va con ellas.


## 14. Errores comunes

- **Usar el acierto con clases raras.** Premia ignorar la clase rara (decir «nunca hay fraude» acierta el
  92%). Mira el coste, o métricas por clase como recall y precisión.
- **Quedarse con el umbral 0,5.** Es solo el centro de la escala, no tiene nada de especial. Elígelo por
  coste.
- **Olvidar que el umbral depende del negocio.** No hay un umbral «correcto» universal: el mismo modelo pide
  umbrales distintos en fraude, spam o cribado médico.
- **Mirar solo el lado caro.** Bajar el umbral caza más fraudes, pero dispara las falsas alarmas; el óptimo
  equilibra los dos costes, no minimiza uno solo.
- **Fiarte de probabilidades mal calibradas** para elegir el umbral. Si están infladas, el umbral óptimo que
  calcules estará sesgado: calibra primero.


## 15. Qué te llevas

- El **acierto** es mala guía cuando una clase es rara y un error cuesta más que otro: con un 8% de fraude,
  decir «nunca hay fraude» acierta el 92% y es inútil.
- La **matriz de confusión** separa los dos errores; ponerles **precio** convierte la evaluación en una
  cuenta de dinero, y el **umbral** es la palanca para minimizarlo.
- Ese umbral óptimo casi nunca vale 0,5, tiene una **fórmula** ($C_{FP}/(C_{FP}+C_{FN})$) cuando las
  probabilidades son honestas, y **se mueve con los costes** y con el problema (fraude, spam, cribado).
- Elegir el umbral es una **decisión de negocio** informada por datos, no un ajuste técnico; y todo el
  cálculo se apoya en que las probabilidades estén **calibradas**.

**Para seguir:** el cuaderno de calibración comprueba si ese «0,8» del modelo es de verdad un 80%; el
facsímil 9 lleva estas decisiones al terreno del riesgo y la gobernanza.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*